# Homework 9: least squares, comparing fits, and outliers

Covers Lecture 22 (`minimize` on a least-squares objective, with `curve_fit` as a check), Lecture 23 (`rmse` and comparing model forms against a measured test) and Lecture 24 (outliers, mean absolute error with `minimize` and refitting on selected rows).

## Submission instructions

Upload the `ipynb` file to Canvas:
> File -> Download -> ipynb -> upload to Canvas (like any other file)

The `ipynb` file type is the only accepted format.

Save a copy before beginning.
Reloading or disconnecting can revert the notebook to the instructor copy.

# Problem 0 (-10 pts if blank): generative AI usage statement

As you work on this assignment, feel free to use generative AI tools to help you learn,
understand, and debug Python code. In particular, you could get hints or conceptual guidance in
the implementation you write yourself.

However, you must clearly disclose and cite all use of AI. You must include:
1. The name(s) of the AI tool(s) used.
2. The specific prompt(s) you used to generate the content.
3. A description of how you used the output and what edits or additions you made to integrate it
   into your own work.

You are fully responsible for the final submitted work. Critically evaluate, fact-check, and
verify all AI-generated content for validity. Failure to properly cite and disclose AI use
constitutes plagiarism under Penn State's Academic Integrity policy.

Write your disclosure (or "I did not use an AI tool for this assignment") in the cell below.

*your disclosure here*

# Data file for this assignment

Run this cell to load the data.
It holds 915 tensile tests on 95 low-alloy steels, each tested at about ten temperatures from 27 °C to 650 °C.

- `Alloy code`: the steel
- `Temperature (°C)`: test temperature
- `0.2% Proof Stress (MPa)`: stress at 0.2% permanent deformation
- `Tensile Strength (MPa)`: highest stress reached

*Data: NIMS [MatNavi](https://mits.nims.go.jp/en/), tensile tests on low-alloy steels.*

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, stats

steels = pd.read_csv('https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/steels.csv')

temp_col = ' Temperature (°C)'                # note the leading space
proof_col = ' 0.2% Proof Stress (MPa)'        # note the leading space
tensile_col = ' Tensile Strength (MPa)'       # note the leading space
steels.head()

,Alloy code,C,Si,Mn,P,S,Ni,Cr,Mo,Cu,V,Al,N,Ceq,Nb + Ta,Temperature (°C),0.2% Proof Stress (MPa),Tensile Strength (MPa),Elongation (%),Reduction in Area (%)
0,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,0.0,0.003,0.0066,0.0,0.0,27,342,490,30,71
1,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,0.0,0.003,0.0066,0.0,0.0,100,338,454,27,72
2,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,0.0,0.003,0.0066,0.0,0.0,200,337,465,23,69
3,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,0.0,0.003,0.0066,0.0,0.0,300,346,495,21,70
4,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,0.0,0.003,0.0066,0.0,0.0,400,316,489,26,79


# Problem 1 (25 pts): a least-squares objective and `minimize`

Steel `CbH` was tested at ten temperatures from 27 °C to 600 °C.
Run this cell to select its tests at 500 °C and below, for Problems 1 and 2.

In [2]:
is_cbh = steels['Alloy code'] == 'CbH'         # True on the rows of steel CbH
cbh = steels.loc[is_cbh]                       # all ten CbH tests, 27 to 600 °C
is_fit_range = cbh[temp_col] <= 500            # True on the eight tests we fit
fit_tests = cbh.loc[is_fit_range]

temp = fit_tests[temp_col].to_numpy()          # °C
tensile = fit_tests[tensile_col].to_numpy()    # MPa

(a) Write a function `strength_line(T, m, b)` that returns the straight line $m\,T + b$.
Then write an objective function `sse_cbh(params)` for that line.

- Return the sum of the squared residuals against the selected tests.
- Print its value for a flat line at 500 MPa.

> `sse_cbh` reads the data from the notebook, so its only argument is `params`.

In [3]:
# your code here

(b) Find $m$ and $b$ with `optimize.minimize`.

- Print the fitted `m` and `b`.
- Print the minimum sum of squared residuals.

> Start both parameters at 1.

In [4]:
# your code here

(c) Fit the same line with `optimize.curve_fit`.

- Print the fitted `m` and `b`.
- Print the sum of squared residuals at those values.

In [5]:
# your code here

(d) Compare the results of (b) and (c).

- Say whether the two agree, using the sums of squared residuals.

*your answer here*

# Problem 2 (30 pts): `rmse` and comparing model forms

Compare three forms for `CbH` at 500 °C and below: your line, a quadratic, and $S = c - a\,e^{kT}$.
The test at 600 °C is outside that range and can check each form.

(a) Fit the quadratic $a\,T^2 + b\,T + c$ as `quadratic(T, a, b, c)` and the decline form $c - a\,e^{kT}$ as `strength_decline(T, a, k, c)`, both with `curve_fit`.

- Write a function `rmse(y, y_fit)` that returns the root-mean-square error.
- Print the RMSE of the line, the quadratic, and the decline form.

> For `strength_decline`, use `p0=[1, 0.01, 600]`.

In [6]:
# your code here

(b) Plot the data with all three fits on one chart.
Plot the residuals of the three fits on a second chart.

In [7]:
# your code here

(c) Check each form against the measured test at 600 °C.

- Print the measured tensile strength at 600 °C.
- Print each form's prediction at 600 °C.

In [8]:
# your code here

(d) Choose which of the three forms is best supported for predicting the tensile strength of `CbH` at 600 °C.

- Justify the choice in one or two sentences, using the fitting RMSE, the residuals, and the measured check.

*your answer here*

# Problem 3 (20 pts): `minimize` with a mean absolute error objective

Steel `VbF` was tested at ten temperatures from 27 °C to 650 °C.
One of its tensile strengths looks out of place.

(a) Select the tests of steel `VbF`.

- Plot tensile strength against temperature, with both axes labeled.

In [9]:
# your code here

(b) Fit a line to all ten tests by least squares, and again by minimizing the mean absolute error.

- Write an objective function `mae_vbf(params)` that returns the mean of the absolute residuals.
- Print both slopes.

> Use `method='Nelder-Mead'`, as in Lecture 24, and start both parameters at 1.

In [10]:
# your code here

(c) Repeat (b) on the nine tests other than the one at 300 °C.

- Print both slopes.
- Plot the ten tests with all four lines, solid for the ten-test fits and dashed for the nine-test fits, with labeled axes and a legend.

> `mae_vbf` reads all ten tests, so the nine-test fit needs its own objective function.
>
> Use `ax.set_ylim(0, 2000)` so the lines are visible. The 300 °C test is above this view.

In [11]:
# your code here

(d) Compare each method's slope for ten tests with its slope for nine tests.

- Say which method's slope moves more.
- Say why, in one or two sentences.

*your answer here*

# Problem 4 (25 pts): selecting rows with a mask and refitting

Use all 915 tests to examine how one record affects a pooled fit.

(a) Select the tests with a tensile strength above 2000 MPa, using a boolean mask.
This is the test from Problem 3.

- Print their alloy code, temperature, proof stress, and tensile strength.

In [12]:
# your code here

(b) Fit a line of tensile strength against temperature to all 915 tests.
Then refit without the test from (a).

- Print the slope and R² of each fit.
- Plot all 915 tests with both lines, with labeled axes and a legend.

> Use `ax.set_ylim(0, 1000)`. The suspect test is above this view.

In [13]:
# your code here

(c) Compare the effect of excluding this test on `VbF` alone and on the pooled table.

- In your plot from (b), the two pooled lines nearly overlap.
  In your plot from 3(c), the two least-squares lines are far apart.
  Say why, in one or two sentences.
- Recommend how to report the suspect test, using its proof stress and your plot from Problem 3 as evidence.

*your answer here*